# Create KFP scoring pipeline and Schedule run

## *Content* 


- [1 Setup and load local pkgs](#1-Setup-and-load-local-pkgs)
- [2 Create task and setup for AI platform](#2-Create-task-and-setup-for-AI-platform)
    - [2.1 task entrypoint](#2.1-task-entrypoint)
    - [2.2 setup for dist](#2.2-setup-for-dist)
    - [2.3 create tar.gz file](#2.3-create-tar.gz-file)
    - [2.4 copy to gcs](#2.4-copy-to-gcs)
- [3 Config AI platform and run job](#4-Config-AI-platform-and-run-job)
    - [3.1 config the job](#4.1-config-the-job)
    - [3.2 submit the job](#4.2-submit-the-job)

## 1 Setup and load local pkgs

In [3]:
# load the libraries and initial settings
import os
import sys
import time
import pathlib
from google.cloud import storage
import google.cloud.aiplatform as aiplatform
import datetime
import pytz
import subprocess
import shutil
import numpy as np
import pandas as pd
cur_path = pathlib.Path().resolve().parent.absolute()
src_loc = cur_path.joinpath("model_package","model_scoring")
util_loc = cur_path.joinpath("model_package")
sys.path.append(str(cur_path))
sys.path.append(str(src_loc))
sys.path.append(str(util_loc))
import warnings
warnings.filterwarnings('ignore')

from gcp_utils.gcs_utils import download_blob, upload_blob, list_blobs

In [4]:
# use 'user' and 's******' for testing
# use 'service' and None for releasing
from config import config
module_name = 'model_scoring'
module_version = '0.0.1'
param_model_id='amm-395'
param_principle='user'   #'service'
param_s_number= 's123815' # None
project_name = 'ctp-syndicate-detection'
params = config(principle=param_principle, model_id=param_model_id, s_number=param_s_number)

In [5]:
script_path = os.path.dirname(os.path.realpath(sys.argv[0]))
print(script_path)

C:\workspace\contentious_claim_all_brands\.venv\Lib\site-packages


## 2 Create task and setup for AI platform

### 2.1 task entrypoint

In [7]:
# get scecrets key
secret_key = "aap-prod-model-secret" if param_s_number is None else f"aap-prod-{param_s_number}-secret"
script_content = f"""#!/bin/bash
export NO_PROXY="secretmanager.googleapis.com,metadata.google.internal"
PROJECT_ID="dia-frd-0586"
gcloud config set core/custom_ca_certs_file /etc/ssl/certs/ca-certificates.crt
gcloud config set project $PROJECT_ID
gcloud secrets versions access "latest" --secret="{secret_key}" > ./secrets.json

# Correctly handling single to double quotes conversion
sed -i 's/'\\''/"/g' ./secrets.json

if [[ $? -eq 0 ]]; then
  echo "INFO: EDH Credentials set"
else
  echo "WARN: Failed to get EDH credentials"
fi
"""

# Write to a .sh file
file_path = os.path.join(src_loc, "get_edh_secrets.sh")
with open(file_path, "w") as file:
    file.write(script_content)

In [61]:
# create setup to install dependencies when ai platform runs a training job
# Specify the file path
file_path = os.path.join(src_loc, "install_packages.sh")

# Define the content to write to the file
content = '''
#!/bin/bash
set -e

cp proxy.conf /etc/apt/apt.conf.d/proxy.conf

# # Setting proxy ca certificates
export REQUESTS_CA_BUNDLE="/etc/ssl/certs/ca-certificates.crt"
export SSL_CERT_FILE="/etc/ssl/certs/ca-certificates.crt"
export CURL_CA_BUNDLE="/etc/ssl/certs/ca-certificates.crt"
export GRPC_DEFAULT_SSL_ROOTS_FILE_PATH="/etc/ssl/certs/ca-certificates.crt"
export NODE_EXTRA_CA_CERTS="/etc/ssl/certs/ca-certificates.crt"

cp iag_ca_cert_chain.pem /usr/local/share/ca-certificates/ca-bundle.crt
update-ca-certificates


python -m pip install --index-url https://nexus3.auiag.corp/repos/repository/ddo-pypi/simple --trusted-host nexus3.auiag.corp -r requirements.txt
'''

# Open the file in write mode and write the content
with open(file_path, "w") as file:
    file.write(content)

### 2.3 setup for dist

In [18]:
# create setup to install dependencies when ai platform runs a training job
# Specify the file path
file_path = os.path.join(util_loc, "setup.py")

# Define the content to write to the file
content = '''
from setuptools import find_packages
from setuptools import setup
import os, sys

module_name = 'model_scoring'
module_version = '0.0.1'

# Get the directory of the setup script

setup(
    name=module_name,
    version=module_version,
    description='data extraction, transformation, model scoring, and data writing',
    # packages=find_packages(include=[f"{module_name}", f"{module_name}.*"], where=str(cur_dir)),
    packages=[module_name, f"{module_name}.utils"],
    package_data={'': [os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/proxy.conf',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/iag_ca_cert_chain.pem',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/get_edh_secrets.sh',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/requirements.txt',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/install_packages.sh',
                      # os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/**/*',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/cert/*',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/data/*',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/queries/*',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/trained_models/*',
                      os.path.dirname(os.path.realpath(__file__))+f'/{module_name}/utils/*',
]},
    include_package_data=True,
    exclude_package_data={'': ['secrets.json'],'model_scoring': ['secrets.json'],'': ['**/secrets.json']},
    install_requires=[],
)
'''

# Open the file in write mode and write the content
with open(file_path, "w") as file:
    file.write(content)

### 2.4 create tar.gz file

In [41]:
# run setup.py sdist
# generate metadata in egg-info which contains 
# setuptools will creates a source distribution archive (a .tar.gz file) 
module_name = 'model_scoring'
module_version = '0.0.1'
notebook_path = os.getcwd()
# Remove old python package files if any
dist_dir = os.path.join(util_loc, "dist")
egg_info_dir = os.path.join(util_loc, f"{module_name}.egg-info")

if os.path.exists(dist_dir):
    shutil.rmtree(dist_dir)

if os.path.exists(egg_info_dir):
    shutil.rmtree(egg_info_dir)
os.chdir(util_loc)
# Create Python Source Code distribution
subprocess.run(["python", "setup.py", "sdist", 
                "--dist-dir", dist_dir, "--formats", "gztar"])
# Change back to the original directory
os.chdir(notebook_path)
# run it from terminal
# python run_setup.py
# python setup.py sdist --dist-dir=dist --formats=gztar

### 2.5 copy to gcs

In [20]:
%load_ext autoreload
%autoreload 2

In [42]:
upload_blob(params['bucket_name'], params["model_python_package_local_file_path"], params["model_python_package_gcs_blob_name"])

File C:\workspace\CTP_syndicate_model\model_package/dist\model_scoring-0.0.1.tar.gz uploaded to amm-395/code/package/model_scoring/model_scoring-0.0.1.tar.gz.


In [43]:
list_blobs(params['bucket_name'], param_model_id)

amm-395/code/package/model_scoring/model_scoring-0.0.1.tar.gz
amm-395/pipelines/amm-395-pipeline.json


## 3 Config AI platform and run job

### 3.1 config the job

In [27]:
from kfp.v2 import compiler, dsl
from kfp.v2.dsl import component, pipeline, Artifact, ClassificationMetrics, Input, Output, Model, Metrics

In [62]:
# define components using params
@component(base_image=params['model_customjob_base_image'])
# def scoring_data_extract_component(config: dict = None):
def create_scoring_component(model_python_package_name: str, 
                             model_python_package_gcs_blob_name: str, 
                             bucket_name: str,
                             entry_py: str):

        """Component to run the data extraction for scoring.
        """
        import os
        import site
        import subprocess

        from google.cloud import storage
        import logging
        import importlib
        import sys
        
        # 1. Download the model package from GCS
        blob = storage.Client().bucket(bucket_name).blob(model_python_package_gcs_blob_name)
        local_file = './' + os.path.basename(model_python_package_gcs_blob_name)
        blob.download_to_filename(local_file)

        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--no-deps", local_file,
            "--index-url", "https://nexus3.auiag.corp/repos/repository/ddo-pypi/simple",
            "--trusted-host", "nexus3.auiag.corp",
        ])
        
        # 2. Locate package install path
        site_packages = site.getsitepackages()[0]  # or use site.getusersitepackages()
        package_path = os.path.join(site_packages, model_python_package_name)

        if os.path.isdir(package_path):
            os.chdir(package_path)
            print(f"Changed directory to: {os.getcwd()}")
        else:
            raise FileNotFoundError(f"Package not found: {package_path}")
        # Set up env with updated PYTHONPATH
        model_package_path = os.path.join(package_path, "model_package")
        env = os.environ.copy()
        env["PYTHONPATH"] = f"{package_path}:{model_package_path}:{env.get('PYTHONPATH', '')}"

        # 3. Change working directory to package root and run build.sh
        # os.chdir(package_path)
        for script in ['install_packages.sh', 'get_edh_secrets.sh']:
            if not os.path.exists(script):
                raise FileNotFoundError(f"Script not found: {script}")
            subprocess.check_call(['chmod', '+x', script])
            # subprocess.run([f'./{script}'])
            subprocess.run(['bash', script], check=True)

        # 4. Run the main training module # could be model_package.model_scoring.predict
        subprocess.run(["python", "-m", entry_py], env=env)

In [64]:
# create pipeline
@pipeline(
    name=f"{param_model_id} pipeline",
    description="A pipeline to run the scoring code",
)
def model_pipeline(model_python_package_name: str,
                   model_python_package_gcs_blob_name: str,
                   bucket_name: str,
                   op_extract: str = "utils.data_extraction.data_extraction",
                   op_transform: str = "utils.data_transformation.data_transformation",
                   op_construct: str = "utils.community_detection.graph_construction",
                   op_inference: str = "utils.community_detection.model_scoring",
                   op_output: str = "utils.data_output.edh_writing"):
    
    op_extract_step = create_scoring_component(
        model_python_package_name=model_python_package_name,
        model_python_package_gcs_blob_name=model_python_package_gcs_blob_name,
        bucket_name=bucket_name,
        entry_py=op_extract
    ).set_cpu_limit('8').set_memory_limit('64G').set_display_name("Data Extraction")

    op_transform_step = create_scoring_component(
        model_python_package_name=model_python_package_name,
        model_python_package_gcs_blob_name=model_python_package_gcs_blob_name,
        bucket_name=bucket_name,
        entry_py=op_transform
    ).set_cpu_limit('8').set_memory_limit('64G').set_display_name("Data Transformation").after(op_extract_step)

    op_construct_step = create_scoring_component(
        model_python_package_name=model_python_package_name,
        model_python_package_gcs_blob_name=model_python_package_gcs_blob_name,
        bucket_name=bucket_name,
        entry_py=op_construct
    ).set_cpu_limit('8').set_memory_limit('64G').set_display_name("Graph Construction").after(op_transform_step)

    op_inference_step = create_scoring_component(
        model_python_package_name=model_python_package_name,
        model_python_package_gcs_blob_name=model_python_package_gcs_blob_name,
        bucket_name=bucket_name,
        entry_py=op_inference
    ).set_cpu_limit('8').set_memory_limit('64G').set_display_name("Model Scoring").after(op_construct_step)

    op_output_step = create_scoring_component(
        model_python_package_name=model_python_package_name,
        model_python_package_gcs_blob_name=model_python_package_gcs_blob_name,
        bucket_name=bucket_name,
        entry_py=op_output
    ).set_cpu_limit('8').set_memory_limit('64G').set_display_name("EDH Writing").after(op_inference_step)

In [65]:
compiler.Compiler().compile(
    pipeline_func=model_pipeline, package_path=params["pipeline_file_name"]
)

In [66]:

with open(params["pipeline_file_name"], "r") as f:
    content = f.read()

content = content.replace("--quiet", "--quiet --index-url https://nexus3.auiag.corp/repos/repository/ddo-pypi/simple --trusted-host nexus3.auiag.corp")

with open(params["pipeline_file_name"], "w") as f:
    f.write(content)


In [67]:
upload_blob(params['bucket_name'], params["pipeline_file_name"], params["pipeline_gcs_blob_name"])

File amm-395-pipeline.json uploaded to amm-395/pipelines/amm-395-pipeline.json.


In [69]:
pipeline_job = aiplatform.PipelineJob(
    display_name=params['pipeline_job_display_name'],
    template_path=params['pipeline_file_name'],
    pipeline_root=params['pipeline_gcs_root'],
    # job_id=params['pipeline_job_id'],
    job_id = 'amm-test-2',
    enable_caching=False,
    encryption_spec_key_name=params['cmek'],
    labels={"a-version": "0_0_1", 
                "b-project": "ctp-syndicate-detection"},
    location=params['region'],
    project=params['project_id'],
    parameter_values={
        'model_python_package_name': params['model_python_package_name'], 
        'model_python_package_gcs_blob_name': params['model_python_package_gcs_blob_name'],
        'bucket_name': params['bucket_name']
    },
)


### 3.2 submit the job

In [70]:
pipeline_job.submit(service_account=params['service_account'], network=params['iag_network'])

Creating PipelineJob
PipelineJob created. Resource name: projects/273950633917/locations/australia-southeast1/pipelineJobs/amm-test-2
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/273950633917/locations/australia-southeast1/pipelineJobs/amm-test-2')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/australia-southeast1/pipelines/runs/amm-test-2?project=273950633917


In [17]:
# %% scheduling the job
import datetime
start_date = (datetime.datetime.now(pytz.timezone('Australia/Sydney')) + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
pipeline_schedule = pipeline_job.create_schedule(
    cron="CRON_TZ=Australia/Sydney 00 20 * * *",
    display_name=params['pipeline_job_display_name']+"-daily-schedule",
    start_time=f"{start_date}T00:00:00Z",  # Start date 
    service_account=params['service_account'],
    network=params['iag_network'],
)

Creating PipelineJobSchedule
PipelineJobSchedule created. Resource name: projects/3109092682/locations/australia-southeast1/schedules/7889426937850888192
To use this PipelineJobSchedule in another session:
schedule = aiplatform.PipelineJobSchedule.get('projects/3109092682/locations/australia-southeast1/schedules/7889426937850888192')
View Schedule:
https://console.cloud.google.com/vertex-ai/locations/australia-southeast1/pipelines/schedules/7889426937850888192?project=3109092682


In [5]:
params

{'monitor_python_package_name': 'model_monitoring',
 'monitor_python_package_version': '0.0.1',
 'monitor_python_package_local_file_path': '/home/jupyter/AMM-331_Motor-Manual-Referral-Prioritization/monitor_package/dist/model_monitoring-0.0.1.tar.gz',
 'monitor_python_package_gcs_blob_name': 'amm-331/code/package/model_monitoring/model_monitoring-0.0.1.tar.gz',
 'monitor_report_gcs_path': 'amm-331/monitor/0.0.1',
 'monitor_customjob_display_name': 'amm-331-model-monitoring',
 'monitor_customjob_base_image': 'asia-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0:latest',
 'model_python_package_name': 'model_scoring',
 'model_python_package_version': '0.0.1',
 'model_python_package_local_file_path': '/home/jupyter/AMM-331_Motor-Manual-Referral-Prioritization/model_package/dist/model_scoring-0.0.1.tar.gz',
 'model_python_package_gcs_blob_name': 'amm-331/code/package/model_scoring/model_scoring-0.0.1.tar.gz',
 'model_customjob_display_name': 'amm-331-model-scoring',
 'model_customjob_base